In [2]:
# chain --> to crete pipeline ( output of on become input for others )
# typeof chain --> sequential , parallel , conditional

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

c:\Users\singh\Let's Gooooo\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Mahatma Gandhi is widely considered the "Father of India."'

In [4]:
# Sequntial chain
template1 = PromptTemplate(
    template= "write a deatailed report on the {topic}",
    input_variables= ['topic']
    )

template2 = PromptTemplate(
    template= "Give me 5 point summary for this given text: \n {text}",
    input_variables= ['text']
    )

parser = StrOutputParser()

chain = template1 | llm_gemini | parser | template2 | llm_gemini | parser

print(chain.invoke("Data Scientist Field"))
# print(chain.get_graph().print_ascii())

Here's a 5-point summary of the provided text:

*   **Data Science Defined:** Data science is an interdisciplinary field using scientific methods to extract knowledge and insights from data, emphasizing prediction and optimization beyond traditional business intelligence.
*   **Essential Skill Set:** Successful data scientists need technical skills (programming, databases, machine learning), analytical skills (statistical analysis, problem-solving), and soft skills (communication, collaboration, business acumen).
*   **Varied Roles and Responsibilities:** Data scientist roles range from data collection and preparation to model deployment and communication, with specializations like Machine Learning Engineer, Data Analyst, and Data Engineer.
*   **Education and Training Pathways:** Entry into data science can be achieved through formal education (degrees), bootcamps, online courses, and self-learning, with a strong portfolio being crucial.
*   **Opportunities, Challenges, and Future Tre

In [5]:
# Building parallel chains
from langchain.schema.runnable import RunnableParallel

parser = StrOutputParser()

template1 = PromptTemplate(
    template="give me short and sharp notes for the txt i'm giving you \n {text}",
    input_variables=['text']
)
template2 = PromptTemplate(
    template= "give me 5 question with answer from thr following text \n {text}",
    input_variables= ['text']
)
template3 = PromptTemplate(
    template= 'combine the provide notes and document in the same document \n notes : {notes} \n quiz : {quiz}',
    input_variables=['notes' , 'quiz']
)

parallel_chain = RunnableParallel({
    "notes" : template1 | llm_gemini | parser,
    "quiz" : template2 | llm_gemini | parser
})

final_chain = parallel_chain | template3 | llm_gemini | parser

text = """
Qwen3-480B-A35B-Instruct has the following features:

Type: Causal Language Models
Training Stage: Pretraining & Post-training
Number of Parameters: 480B in total and 35B activated
Number of Layers: 62
Number of Attention Heads (GQA): 96 for Q and 8 for KV
Number of Experts: 160
Number of Activated Experts: 8
Context Length: 262,144 natively.
NOTE: This model supports only non-thinking mode and does not generate <think></think> blocks in its output. Meanwhile, specifying enable_thinking=False is no longer required.

For more details, including benchmark evaluation, hardware requirements, and inference performance, please refer to our blog, GitHub, and Documentation.

Quickstart
We advise you to use the latest version of transformers.

With transformers<4.51.0, you will encounter the following error:

KeyError: 'qwen3_moe'
"""
print(final_chain.invoke(text))

# Qwen3-480B-A35B-Instruct: Model Overview & Quiz

This document combines quick reference notes on the Qwen3-480B-A35B-Instruct model with a brief quiz to test comprehension.

## Qwen3-480B-A35B-Instruct: Quick Notes

*   **Type:** Causal Language Model
*   **Training:** Pretrained & Post-trained
*   **Parameters:** 480B (35B activated)
*   **Layers:** 62
*   **Attention Heads:** Q=96, KV=8 (GQA)
*   **Experts:** 160 (8 activated)
*   **Context:** 262,144
*   **Non-Thinking Mode:** Always on (no `<think>` blocks). `enable_thinking=False` not needed.
*   **Transformers:** Use latest version (>=4.51.0) to avoid "KeyError: 'qwen3_moe'".
*   **Details:** See blog, GitHub, and Documentation.

## Quiz: Qwen3-480B-A35B-Instruct

**Question 1:**

What are the two training stages of the Qwen3-480B-A35B-Instruct model?

**Answer:**

Pretraining and Post-training

**Question 2:**

How many parameters are activated in the Qwen3-480B-A35B-Instruct model?

**Answer:**

35B

**Question 3:**

What is 

In [6]:
final_chain.get_graph().print_ascii()

                    +---------------------------+                      
                    | Parallel<notes,quiz>Input |                      
                    +---------------------------+                      
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati

In [7]:
# Building Conditional chains
from pydantic import BaseModel , Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain.schema.runnable import RunnableBranch , RunnableLambda

class Classifiction(BaseModel):
    sentiment : Literal['Postitive' , 'Negative'] = Field(description= "sentiment of the given text")

pydantic_parser = PydanticOutputParser(pydantic_object=Classifiction)

template1 = PromptTemplate(
    template= "Classify the follwoing text into a nagative or positive sentiment \n {text} and {format_intstr}",
    input_variables= ['text'],
    partial_variables= {'format_intstr' : pydantic_parser.get_format_instructions()}
)

template2 = PromptTemplate(
    template= "write me an appopirate Resnponse to this postive text : \n {text}give me one final text, that i can direclty show to the user",
    input_variables= ['text']
)

template2 = PromptTemplate(
    template= "write me an appopirate Resnponse to this Negative text : \n {text} that i can direclty show to the user",
    input_variables= ['text']
)

classifier_chain = template1 | llm_gemini | pydantic_parser

branch_chain = RunnableBranch(
    (lambda x : x.sentiment == 'Postitive' , template2 | llm_gemini | parser),
    (lambda x : x.sentiment == 'Negative' , template3 | llm_gemini | parser),
     RunnableLambda(lambda x: "could not find sentiment")
)

final_chain = classifier_chain | branch_chain
print(final_chain.invoke("this is a Very Good mobile pphone"))

Okay, here are a few response options, ranging in tone, that you can use when your system incorrectly identifies a negative sentiment as positive. Choose the one that best fits your brand and the context of the user's message:

**Option 1 (Acknowledging the potential error, neutral tone):**

> "Thanks for your feedback. Our system flagged this as positive, but we appreciate you bringing it to our attention. We're always working to improve our understanding of different perspectives."

**Option 2 (More direct, focused on learning):**

> "Thank you for pointing that out. Our sentiment analysis seems to be off on this one. We'll use this example to help us learn and improve."

**Option 3 (Slightly apologetic, but still professional):**

> "Oops! Our apologies, it seems like we misinterpreted your message. Thanks for letting us know. We'll make sure to review this and improve our system."

**Option 4 (More conversational and reassuring):**

> "Thanks for the heads up! Our system thought th

In [8]:
final_chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
